Let's repeat what we did in the previous notebook (import libraries and clean the data).

In [14]:
#Importing necessary libraries 
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as mpl
import seaborn as sns

#Printing library versions for user clarification
print(f"NumPy version: {np.__version__}")
print(f"pandas version: {pd.__version__}")
print(f"matplotlib version: {matplotlib.__version__}")
print(f"seaborn version: {sns.__version__}")

data = pd.read_csv("../data/players.csv")
gk_columns = data.columns[range(54,72,1)]
data = data.drop(columns=gk_columns)
data = data.dropna(subset=["goals", "minutes", "assists"])
data = data.drop(columns=["pens_won", "pens_conceded"])

#Keep only numeric data
data = data.select_dtypes(include='number')

NumPy version: 2.5.1
pandas version: 3.0.5
matplotlib version: 3.11.1
seaborn version: 0.13.2


In [15]:
data.head()

,age,birth_year,games,games_starts,minutes,minutes_90s,goals,assists,goals_assists,goals_pens,...,plus_minus_per90,plus_minus_wowy,cards_yellow_red,fouls,fouled,offsides,crosses,interceptions,tackles_won,own_goals
1,23,2003,2,0,19.0,0.2,0.0,0.0,0.0,0.0,...,0.00,1.06,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,26,2000,4,3,251.0,2.8,1.0,0.0,1.0,1.0,...,-0.36,2.12,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0
3,24,2002,3,1,98.0,1.1,0.0,0.0,0.0,0.0,...,-0.92,0.11,0.0,0.0,2.0,0.0,5.0,1.0,0.0,0.0
4,34,1991,4,4,360.0,4.0,0.0,0.0,0.0,0.0,...,-1.00,NaN,0.0,0.0,5.0,0.0,0.0,4.0,3.0,0.0
5,23,2002,4,4,360.0,4.0,0.0,0.0,0.0,0.0,...,-1.00,NaN,0.0,3.0,2.0,1.0,6.0,3.0,5.0,0.0


We are going to use three different models and compare them to each other to see which will most accurately predict the number of goals scored by a player. 

1. Linear Regressor - We model the number of goals as a linear function of the other features 
2. Random Forest Regressor - We use a combination of different decision trees to predict number of goals
3. Poisson Regressor - We model the number of goals as a function of the other features and assume a poisson distribution
 
To do this, we split our dataset into training and validation sets, and compare the accuracy which will be calculated after cross-validation to see how effective our models are.

First, let's define our features and target variable. Since it would be quite easy to predict the number of goals given the number of assists and the number of goals+assists, we should remove all columns which encompass the number of goals in them so we can actually take some meaningful insight from our models.

In [16]:
#Create a mask which is true for all columns which contain the word "goals" 
#and then apply it to the dataset, keeping only columns which are false in the mask
X = data.loc[:, ~data.columns.str.contains("goals", case=False)]
X.head()

,age,birth_year,games,games_starts,minutes,minutes_90s,assists,pens_made,pens_att,cards_yellow,...,plus_minus,plus_minus_per90,plus_minus_wowy,cards_yellow_red,fouls,fouled,offsides,crosses,interceptions,tackles_won
1,23,2003,2,0,19.0,0.2,0.0,0.0,0.0,0.0,...,0.0,0.00,1.06,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,26,2000,4,3,251.0,2.8,0.0,0.0,0.0,0.0,...,-1.0,-0.36,2.12,0.0,1.0,1.0,0.0,1.0,0.0,1.0
3,24,2002,3,1,98.0,1.1,0.0,0.0,0.0,0.0,...,-1.0,-0.92,0.11,0.0,0.0,2.0,0.0,5.0,1.0,0.0
4,34,1991,4,4,360.0,4.0,0.0,0.0,0.0,0.0,...,-4.0,-1.00,NaN,0.0,0.0,5.0,0.0,0.0,4.0,3.0
5,23,2002,4,4,360.0,4.0,0.0,0.0,0.0,1.0,...,-4.0,-1.00,NaN,0.0,3.0,2.0,1.0,6.0,3.0,5.0


In [17]:
y = data.goals
y

1       0.0
2       1.0
3       0.0
4       0.0
5       0.0
       ... 
1242    0.0
1243    0.0
1244    0.0
1245    0.0
1247    0.0
Name: goals, Length: 1039, dtype: float64